# Sanity Check - Step 08: Interbrain PLV (Phase-Locking Value)

Überprüft:
- PLV korrekt zwischen P1 und P2 berechnet
- Frequenzbande korrekt (Alpha: 8-12 Hz)
- PLV Wertebereich [0, 1]
- Kanal-zu-Kanal Synchronisierungsmuster

In [ ]:
import sys
import mne
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent / 'eeg_pipeline'))
import config

print("Setup erfolgreich")

## 1. Input-Daten laden: Epochen von Step 06

In [ ]:
subject_id = config.SUBJECTS[0]

# Step 06 Output laden (Epochen)
p1_path = config.OUTPUT_DIR / f"sub-{subject_id}_P1_epoch.fif"
p2_path = config.OUTPUT_DIR / f"sub-{subject_id}_P2_epoch.fif"

epochs_p1 = mne.read_epochs(str(p1_path), preload=True)
epochs_p2 = mne.read_epochs(str(p2_path), preload=True)

print(f"\n=== INPUT EPOCHEN (Step 06 Output) ===\n")
print(f"Person 1:")
print(f"  Anzahl Epochen: {len(epochs_p1)}")
print(f"  EEG Kanäle: {len(mne.pick_types(epochs_p1.info, eeg=True))}")
print(f"  Kanal-Namen: {mne.pick_types(epochs_p1.info, eeg=True, ret_names=True)[0][:5]}...")
print(f"  Sampling Rate: {epochs_p1.info['sfreq']:.0f} Hz")
print(f"  Epoch-Länge: {epochs_p1.times[-1] - epochs_p1.times[0]:.3f} s")

print(f"\nPerson 2:")
print(f"  Anzahl Epochen: {len(epochs_p2)}")
print(f"  EEG Kanäle: {len(mne.pick_types(epochs_p2.info, eeg=True))}")
print(f"  Kanal-Namen: {mne.pick_types(epochs_p2.info, eeg=True, ret_names=True)[0][:5]}...")
print(f"  Sampling Rate: {epochs_p2.info['sfreq']:.0f} Hz")
print(f"  Epoch-Länge: {epochs_p2.times[-1] - epochs_p2.times[0]:.3f} s")

In [ ]:
# Kanal-Vergleich prüfen
print(f"\n=== KANAL-VERGLEICH ===\n")

ch_p1 = mne.pick_types(epochs_p1.info, eeg=True, ret_names=True)[0]
ch_p2 = mne.pick_types(epochs_p2.info, eeg=True, ret_names=True)[0]

if ch_p1 == ch_p2:
    print(f"✓ Kanal-Namen identisch")
    print(f"  Kanäle: {ch_p1}")
else:
    print(f"✗ WARNUNG: Kanal-Namen unterschiedlich!")
    print(f"  P1: {ch_p1}")
    print(f"  P2: {ch_p2}")

if epochs_p1.info['sfreq'] == epochs_p2.info['sfreq']:
    print(f"✓ Sampling Rates identisch ({epochs_p1.info['sfreq']:.0f} Hz)")
else:
    print(f"✗ WARNUNG: Sampling Rates unterschiedlich!")
    print(f"  P1: {epochs_p1.info['sfreq']:.0f} Hz")
    print(f"  P2: {epochs_p2.info['sfreq']:.0f} Hz")

In [ ]:
# PLV Parameter aus config
print(f"\n=== PLV PARAMETER ===\n")

print(f"Konfigurierte Parameter:")
print(f"  Frequenz-Bereich (IBS): {config.IBS_FMIN} - {config.IBS_FMAX} Hz (Alpha Band)")
print(f"  Sampling Rate: {epochs_p1.info['sfreq']:.0f} Hz")
print(f"  Anzahl Epochen (P1): {len(epochs_p1)}")
print(f"  Anzahl Epochen (P2): {len(epochs_p2)}")
print(f"  Min Epochen (wird verwendet): {min(len(epochs_p1), len(epochs_p2))}")

In [ ]:
# Berechne PLV
from mne_connectivity import spectral_connectivity_epochs

# Pick nur EEG Kanäle
epochs_p1_eeg = epochs_p1.copy().pick_types(eeg=True)
epochs_p2_eeg = epochs_p2.copy().pick_types(eeg=True)

# Equalisiere Epochen-Anzahl
n_epochs = min(len(epochs_p1_eeg), len(epochs_p2_eeg))
epochs_p1_eeg = epochs_p1_eeg[:n_epochs]
epochs_p2_eeg = epochs_p2_eeg[:n_epochs]

print(f"\nBerechne PLV für {n_epochs} Epochen...\n")

# Kombiniere Epochen (P1 und P2)
data_p1 = epochs_p1_eeg.get_data()  # (n_epochs, n_channels, n_times)
data_p2 = epochs_p2_eeg.get_data()

# Stapele die Daten vertikal: (2*n_epochs, n_channels, n_times)
data_stacked = np.vstack([data_p1, data_p2])

# Erstelle künstliche Events für spektrale Konnektivität
events_stacked = np.zeros((2*n_epochs, 3), dtype=int)
events_stacked[:, 0] = np.arange(2*n_epochs)
events_stacked[n_epochs:, 2] = 1  # P2 hat Event-ID 1

# Erstelle Info
info = epochs_p1_eeg.info.copy()

# Berechne PLV
print("Berechne spektrale Konnektivität...")
plv, freqs = spectral_connectivity_epochs(
    (data_stacked, info, events_stacked, epochs_p1_eeg.event_id),
    method='plv',
    mode='multitaper',
    fmin=config.IBS_FMIN,
    fmax=config.IBS_FMAX,
    verbose=False
)

print(f"\nPLV berechnet!")
print(f"  PLV Shape: {plv.shape}")
print(f"  Frequenzen: {freqs}")

In [ ]:
# Vereinfachte manuelle PLV Berechnung als Validierung
from scipy import signal as sig

print(f"\n=== MANUELLE PLV BERECHNUNG ZUR VALIDIERUNG ===\n")

# Nutze nur einen Kanal pro Person für Demo
ch_idx = 15  # Mittlerer Kanal
ch_name = epochs_p1_eeg.ch_names[ch_idx]

print(f"Demonstriere auf Kanal: {ch_name}")

# Extrahiere Signal für einen Kanal
data_p1_ch = epochs_p1_eeg.get_data()[:, ch_idx, :]  # (n_epochs, n_times)
data_p2_ch = epochs_p2_eeg.get_data()[:, ch_idx, :]  # (n_epochs, n_times)

print(f"\nSignal-Eigenschaften:")
print(f"  P1 Signal: min={np.min(data_p1_ch):.6f}, max={np.max(data_p1_ch):.6f}, std={np.std(data_p1_ch):.6f}")
print(f"  P2 Signal: min={np.min(data_p2_ch):.6f}, max={np.max(data_p2_ch):.6f}, std={np.std(data_p2_ch):.6f}")

# Berechne analytisches Signal (Phase) mit Hilbert Transform
print(f"\nBerechne Phase mit Hilbert Transform...")

# Bandpass Filter im Alpha Band
fs = epochs_p1_eeg.info['sfreq']
nyquist = fs / 2
low = config.IBS_FMIN / nyquist
high = config.IBS_FMAX / nyquist

b, a = sig.butter(4, [low, high], btype='band')

data_p1_filtered = np.zeros_like(data_p1_ch)
data_p2_filtered = np.zeros_like(data_p2_ch)

for epoch_idx in range(n_epochs):
    data_p1_filtered[epoch_idx] = sig.filtfilt(b, a, data_p1_ch[epoch_idx])
    data_p2_filtered[epoch_idx] = sig.filtfilt(b, a, data_p2_ch[epoch_idx])

# Hilbert Transform für Phase
analytic_p1 = sig.hilbert(data_p1_filtered, axis=1)
analytic_p2 = sig.hilbert(data_p2_filtered, axis=1)

phase_p1 = np.angle(analytic_p1)
phase_p2 = np.angle(analytic_p2)

# Berechne Phase Locking Value
phase_diff = phase_p1 - phase_p2
plv_manual = np.abs(np.mean(np.exp(1j * phase_diff), axis=1))  # Mean über Epochen

print(f"\nManuelle PLV Ergebnisse für Kanal {ch_name}:")
print(f"  PLV mean: {np.mean(plv_manual):.6f}")
print(f"  PLV std: {np.std(plv_manual):.6f}")
print(f"  PLV range: [{np.min(plv_manual):.6f}, {np.max(plv_manual):.6f}]")
print(f"  PLV alle im [0, 1]? {np.all((plv_manual >= 0) & (plv_manual <= 1))}")

In [ ]:
# Visualisiere Phase von P1 und P2
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Erste Epoche
epoch_idx = 0
times = epochs_p1_eeg.times

ax1 = axes[0]
ax1.plot(times, np.unwrap(phase_p1[epoch_idx]), 'b-', label='P1 Phase', linewidth=1)
ax1.plot(times, np.unwrap(phase_p2[epoch_idx]), 'r-', label='P2 Phase', linewidth=1, alpha=0.7)
ax1.set_ylabel('Phase (Radianten)')
ax1.set_title(f'Phase Verlauf - Kanal {ch_name} - Erste Epoche')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Phase Differenz
ax2 = axes[1]
phase_diff_epoch = np.unwrap(phase_p1[epoch_idx] - phase_p2[epoch_idx])
ax2.plot(times, phase_diff_epoch, 'g-', linewidth=1, label='Phase Differenz (P1 - P2)')
ax2.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
ax2.set_xlabel('Zeit (s)')
ax2.set_ylabel('Phase Differenz (Radianten)')
ax2.set_title(f'Phase Differenz - Kanal {ch_name} - Erste Epoche')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\nPLV für erste Epoche: {plv_manual[0]:.6f}")

In [ ]:
# PLV Statistiken
print(f"\n=== PLV STATISTIKEN ===\n")

print(f"Kanal: {ch_name}")
print(f"Frequenz-Bereich: {config.IBS_FMIN}-{config.IBS_FMAX} Hz (Alpha)")
print(f"Anzahl Epochen: {n_epochs}")

print(f"\nPLV Werte:")
print(f"  Mean: {np.mean(plv_manual):.6f}")
print(f"  Std: {np.std(plv_manual):.6f}")
print(f"  Min: {np.min(plv_manual):.6f}")
print(f"  Max: {np.max(plv_manual):.6f}")
print(f"  Median: {np.median(plv_manual):.6f}")

print(f"\nPlausibilität:")
print(f"  Alle PLV im [0, 1]? {np.all((plv_manual >= 0) & (plv_manual <= 1))}")
print(f"  PLV > 0.3 (hohe Synchronisierung)? {np.sum(plv_manual > 0.3)} Epochen")
print(f"  PLV < 0.1 (niedrige Synchronisierung)? {np.sum(plv_manual < 0.1)} Epochen")

In [ ]:
# PLV Verteilung über alle Epochen
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax1 = axes[0]
ax1.hist(plv_manual, bins=20, color='skyblue', edgecolor='black', alpha=0.7)
ax1.axvline(np.mean(plv_manual), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(plv_manual):.3f}')
ax1.axvline(np.median(plv_manual), color='green', linestyle='--', linewidth=2, label=f'Median: {np.median(plv_manual):.3f}')
ax1.set_xlabel('PLV Wert')
ax1.set_ylabel('Häufigkeit (Anzahl Epochen)')
ax1.set_title(f'PLV Verteilung - Kanal {ch_name}')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# Zeitreihe
ax2 = axes[1]
ax2.plot(plv_manual, 'o-', color='blue', markersize=4, linewidth=1)
ax2.axhline(np.mean(plv_manual), color='red', linestyle='--', linewidth=1, label=f'Mean')
ax2.set_xlabel('Epoche Index')
ax2.set_ylabel('PLV Wert')
ax2.set_title(f'PLV pro Epoche - Kanal {ch_name}')
ax2.set_ylim([0, 1])
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Phase Lag Index (PLI) als Alternative zu PLV
print(f"\n=== ZUSÄTZLICH: PHASE LAG INDEX (PLI) ===\n")

# PLI ist ähnlich PLV aber fokussiert auf konsistente Verzögerungen
pli_manual = np.abs(np.mean(np.sign(np.sin(phase_diff)), axis=1))

print(f"PLI (Phase Lag Index) für Kanal {ch_name}:")
print(f"  Mean: {np.mean(pli_manual):.6f}")
print(f"  Std: {np.std(pli_manual):.6f}")
print(f"  Range: [{np.min(pli_manual):.6f}, {np.max(pli_manual):.6f}]")

print(f"\nPLV vs PLI Vergleich (erste 10 Epochen):")
print(f"{'Epoche':<10} {'PLV':<10} {'PLI':<10}")
print("-" * 30)
for i in range(min(10, n_epochs)):
    print(f"{i:<10} {plv_manual[i]:<10.6f} {pli_manual[i]:<10.6f}")

In [ ]:
print(f"\n=== SANITY CHECK ZUSAMMENFASSUNG ===\n")

checks = []

# 1. Epochen-Anzahl sollte gleich sein
same_n_epochs = len(epochs_p1) == len(epochs_p2)
checks.append(("Gleiche Epochen-Anzahl (P1 & P2)", same_n_epochs))

# 2. Kanäle sollten gleich sein
same_channels = epochs_p1_eeg.ch_names == epochs_p2_eeg.ch_names
checks.append(("Gleiche Kanal-Namen", same_channels))

# 3. Sampling Rates sollten gleich sein
same_sfreq = epochs_p1.info['sfreq'] == epochs_p2.info['sfreq']
checks.append(("Gleiche Sampling Rate", same_sfreq))

# 4. PLV sollte im Bereich [0, 1] sein
plv_range_ok = np.all((plv_manual >= 0) & (plv_manual <= 1))
checks.append(("PLV im Bereich [0, 1]", plv_range_ok))

# 5. PLV sollte nicht konstant 0 oder 1 sein (würde Fehler andeuten)
plv_varies = (np.std(plv_manual) > 0.001) and (np.mean(plv_manual) > 0.01)
checks.append(("PLV variiert (nicht konstant)", plv_varies))

# 6. PLV sollte plausible Werte haben (typisch 0.1-0.4 für echte Daten)
plv_plausible = (np.mean(plv_manual) > 0.01) and (np.mean(plv_manual) < 0.99)
checks.append(("PLV Mittelwert plausibel (0.01-0.99)", plv_plausible))

# 7. Keine NaN oder Inf in PLV
no_nan_inf = not (np.isnan(plv_manual).any() or np.isinf(plv_manual).any())
checks.append(("Keine NaN/Inf in PLV", no_nan_inf))

# 8. Daten sollten nicht constant sein (würde Fehler andeuten)
data_varies_p1 = np.std(data_p1_ch) > 0
data_varies_p2 = np.std(data_p2_ch) > 0
checks.append(("Eingabe-Daten variieren", data_varies_p1 and data_varies_p2))

for check_name, result in checks:
    status = "✓ PASS" if result else "✗ FAIL"
    print(f"{status}: {check_name}")

all_pass = all(result for _, result in checks)
print(f"\n{'='*50}")
if all_pass:
    print("✓ ALLE CHECKS BESTANDEN")
else:
    print("✗ EINIGE CHECKS FEHLGESCHLAGEN")
print(f"{'='*50}")